In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, fbeta_score, precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from datetime import datetime
import random


In [ ]:

class FraudDetector:
    def __init__(self, scale_pos_weight="auto", random_state=42):
        self.scale_pos_weight = scale_pos_weight
        self.random_state = random_state
        self.model = None
        self.preprocessor = None
        self.best_threshold = None
        self.feature_names = [
            'amount', 'credit_score', 'transaction_velocity', 
            'amount_deviation', 'account_age_years',
            'avg_monthly_income', 'risk_score',  # Fixed missing comma here
            'transaction_type', 'location', 'time_of_day',
            'auth_method', 'age_group', 'home_location',
            'account_type', 'mobile_banking_user',
            'employment_status', 'international_activity'
        ]

    def _validate_input(self, X: pd.DataFrame):
        available_features = set(X.columns)
        required_features = set(self.feature_names)
        
        missing = required_features - available_features
        if missing:
            raise ValueError(f"Missing required features: {missing}\n"
                           f"Available features: {list(available_features)}")
        if X.isna().any().any():
            raise ValueError("Input contains NaN values")

    def _create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        # Ensure datetime conversion
        if 'transaction_date' in df.columns:
            df['transaction_date'] = pd.to_datetime(df['transaction_date'])
            df['hour_of_day'] = df['transaction_date'].dt.hour
            df['day_of_week'] = df['transaction_date'].dt.dayofweek
        
        # Create transaction velocity features
        if 'customer_id' in df.columns and 'transaction_date' in df.columns:
            df = df.sort_values(['customer_id', 'transaction_date'])
            df['time_since_last_txn'] = df.groupby('customer_id')['transaction_date'].diff().dt.total_seconds() / 3600
            df['time_since_last_txn'] = df['time_since_last_txn'].fillna(0)
        
        return df

    def fit(self, X: pd.DataFrame, y: pd.Series, tune_hyperparams: bool = True, n_trials: int = 20):
        self._validate_input(X)
        X_processed = self._create_features(X)
        
        if self.scale_pos_weight == "auto":
            self.scale_pos_weight = (y == 0).sum() / max(1, (y == 1).sum())

        numeric_features = [
            'amount', 'credit_score', 'transaction_velocity', 
            'amount_deviation', 'account_age_years',
            'avg_monthly_income', 'risk_score'
        ]
        categorical_features = [
            'transaction_type', 'location', 'time_of_day',
            'auth_method', 'age_group', 'home_location',
            'account_type', 'mobile_banking_user',
            'employment_status', 'international_activity'
        ]
        
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numeric_features),
                ('cat', OneHotEncoder(handle_unknown="ignore"), categorical_features)
            ]
        )

        X_preprocessed = self.preprocessor.fit_transform(X_processed)
        
        X_train, X_val, y_train, y_val = train_test_split(
            X_preprocessed, y, test_size=0.3, stratify=y, random_state=self.random_state
        )

        if tune_hyperparams:
            def objective(trial):
                params = {
                    'max_depth': trial.suggest_int('max_depth', 4, 8),
                    'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
                    'subsample': trial.suggest_float('subsample', 0.7, 0.9),
                    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
                    'gamma': trial.suggest_float('gamma', 0, 0.7),
                    'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
                    'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
                    'min_child_weight': trial.suggest_int('min_child_weight', 1, 20)
                }
                
                model = XGBClassifier(
                    **params,
                    scale_pos_weight=self.scale_pos_weight,
                    random_state=self.random_state,
                    n_estimators=500,
                    early_stopping_rounds=20,
                    eval_metric='aucpr'
                )
                
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=False
                )
                
                pred_probs = model.predict_proba(X_val)[:, 1]
                precision, recall, _ = precision_recall_curve(y_val, pred_probs)
                f2_score = (5 * precision * recall) / (4 * precision + recall + 1e-6)
                return np.max(f2_score)
            
            study = optuna.create_study(direction='maximize')
            study.optimize(objective, n_trials=n_trials)
            best_params = study.best_params
        else:
            best_params = {
                'max_depth': 6,
                'learning_rate': 0.1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'gamma': 0.2,
                'reg_alpha': 0.5,
                'reg_lambda': 0.5,
                'min_child_weight': 3
            }
        
        self.model = Pipeline([
            ('preprocessor', self.preprocessor),
            ('classifier', XGBClassifier(
                **best_params,
                scale_pos_weight=self.scale_pos_weight,
                eval_metric='aucpr',
                random_state=self.random_state,
                n_estimators=500
            ))
        ])
        
        self.model.fit(X_processed, y)
        
        y_proba = self.model.predict_proba(X_processed)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y, y_proba)
        self.best_threshold = thresholds[np.argmax(recalls >= 0.90)]
        
        y_pred = (y_proba >= self.best_threshold).astype(int)
        print("\nValidation Performance:")
        print(classification_report(y, y_pred))
        print(f"Optimal threshold for 90% recall: {self.best_threshold:.4f}")
    
    def predict(self, X: pd.DataFrame, return_proba: bool = False) -> np.ndarray:
        self._validate_input(X)
        X_processed = self._create_features(X)
        
        if return_proba:
            return self.model.predict_proba(X_processed)[:, 1]
        else:
            proba = self.model.predict_proba(X_processed)[:, 1]
            return (proba >= self.best_threshold).astype(int)

    def save(self, path: str):
        joblib.dump(self, path)
        print(f"Model saved to {path}")
    
    @staticmethod
    def load(path: str) -> 'FraudDetector':
        return joblib.load(path)

In [ ]:
if __name__ == "__main__":
    # Load customer data
    customer_df = pd.read_csv("/Users/lalitramanmishra/GlobalIME/global_ime_bank_customers 2.csv")
    
    for customer_id in customer_df["customer_id"].unique():
        print(f"\nProcessing customer {customer_id}...")
        
        try:
            # Load transactions
            transactions = pd.read_csv(
                f"/Users/lalitramanmishra/GlobalIME/segmented_transactions/user_{customer_id}_transactions.csv"
            )
            detector=FraudDetector()
            
            # Check if target column exists
            if 'is_suspicious' not in transactions.columns:
                print(f"Skipping customer {customer_id}: No 'is_suspicious' column")
                continue
                
            # Prepare data - use the feature_names list to select columns
            try:
                X = transactions[detector.feature_names]  # Select only the features we need
                y = transactions['is_suspicious']
            except KeyError as e:
                missing = set(fraud_detector.feature_names) - set(transactions.columns)
                print(f"Skipping customer {customer_id}: Missing features {missing}")
                continue
            
            # Verify we have both fraud and non-fraud cases
            if len(y.unique()) < 2:
                print(f"Skipping customer {customer_id}: Insufficient fraud cases")
                continue
            
            # Train model
            print("Training fraud detection model...")
            fraud_detector = FraudDetector(random_state=42)
            fraud_detector.fit(X, y, tune_hyperparams=True, n_trials=30)
            
            # Save model
            model_path = f"fraud_detector_model_{customer_id}.joblib"
            fraud_detector.save(model_path)
            print(f"Model saved for customer {customer_id}")
            
        except FileNotFoundError:
            print(f"Transaction file not found for customer {customer_id}")
        except Exception as e:
            print(f"Error processing customer {customer_id}: {str(e)}")  # Fixed line